In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import export_graphviz
import pydot

In [2]:
sensors = pd.read_csv('../Cleaned_data/validsensors.csv')
# Create month as number not string
sensors['datetime'] =pd.to_datetime(sensors['datetime'])#.dt.dayofweek
# Keep only data from 2011 onwards
sensors= sensors[sensors['year']>2010]

In [3]:
weather_all_years = pd.read_csv('../Cleaned_data/weather_data_allyears.csv')
weather_all_years['datetime'] = pd.to_datetime(weather_all_years['datetime'])
sensors_with_features = sensors.merge(weather_all_years, on='datetime', how='left')

In [4]:
# Read in public holiday dates
public_holidays = pd.read_csv('../Cleaned_data/publicholidays.csv')
# Convert date to datetime
public_holidays['datetime'] = pd.to_datetime(public_holidays['Date'])
# Rename column to indicate it relates to public holidays, and set values to 1
public_holidays.rename(columns={'Holiday Name':'public_holiday'}, inplace=True)
public_holidays['public_holiday'] = 1
# Drop date column 
public_holidays = public_holidays.drop(['Date'], axis=1)
# Join to sensors data
sensors_with_features = sensors_with_features.merge(public_holidays,how='left', on='datetime')
sensors_with_features['public_holiday'] = sensors_with_features['public_holiday'].fillna(0)

In [5]:
days = pd.get_dummies(sensors_with_features['day'], drop_first = True)
days.columns = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Saturday', 'Sunday']
months = pd.get_dummies(sensors_with_features['month'], drop_first = True)
months.columns = ['January','February', 'March', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
day_of_month = pd.get_dummies(sensors_with_features['mdate'], drop_first = True)
day_of_month=day_of_month.add_prefix('d_')
years = pd.get_dummies(sensors_with_features['year'], drop_first = False)
hours = pd.get_dummies(sensors_with_features['time'], drop_first = False)
hours= hours.add_prefix('h_')

In [6]:
sensors_with_features = pd.concat([sensors_with_features, days, months, day_of_month, years, hours], axis=1)

In [7]:
sensors_with_features=sensors_with_features[sensors_with_features.Temp.notnull()]

In [8]:
sensors_with_features=sensors_with_features.drop(["sensor_id", 'Latitude', 'Longitude', 'location', 'datetime','year','month','mdate','day','time'], axis=1)

In [47]:
features = sensors_with_features.copy()
features.head(5)

,hourly_counts,Temp,Humidity,Pressure,Rain,WindSpeed,public_holiday,Monday,Tuesday,Wednesday,...,h_14,h_15,h_16,h_17,h_18,h_19,h_20,h_21,h_22,h_23
0,241,23.0,57.0,1008.0,0.0,6.0,0.0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,325,23.0,57.0,1008.0,0.0,6.0,0.0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,233,23.0,57.0,1008.0,0.0,6.0,0.0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,2767,23.0,57.0,1008.0,0.0,6.0,0.0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,985,23.0,57.0,1008.0,0.0,6.0,0.0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [48]:
print('The shape of our features is:', features.shape)

The shape of our features is: (700881, 90)


In [49]:
# Descriptive statistics for each column
features.describe()

,hourly_counts,Temp,Humidity,Pressure,Rain,WindSpeed,public_holiday,Monday,Tuesday,Wednesday,...,h_14,h_15,h_16,h_17,h_18,h_19,h_20,h_21,h_22,h_23
count,700881.000000,700881.000000,700881.000000,700881.000000,700881.000000,700881.000000,700881.000000,700881.000000,700881.000000,700881.000000,...,700881.000000,700881.000000,700881.000000,700881.000000,700881.000000,700881.000000,700881.000000,700881.000000,700881.000000,700881.000000
mean,455.818476,14.698204,68.675012,1016.752658,0.035697,12.083365,0.001528,0.143173,0.142728,0.143177,...,0.041722,0.041586,0.041713,0.041742,0.041708,0.041729,0.041763,0.041756,0.041767,0.041775
std,768.713945,6.103156,19.433409,7.498510,0.185533,6.263249,0.039061,0.350249,0.349795,0.350253,...,0.199953,0.199642,0.199933,0.199999,0.199920,0.199969,0.200048,0.200031,0.200057,0.200074
min,0.000000,-2.000000,0.000000,987.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,30.000000,10.500000,55.500000,1012.000000,0.000000,7.500000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,124.000000,14.000000,72.000000,1017.000000,0.000000,11.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,489.000000,18.000000,84.500000,1022.000000,0.000000,15.666667,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,9805.000000,44.000000,100.000000,1039.000000,1.000000,49.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [50]:
# Use numpy to convert to arrays
# Labels are the values we want to predict
labels = np.array(features['hourly_counts'])
# Remove the labels from the features
# axis 1 refers to the columns
features= features.drop('hourly_counts', axis = 1)
# Saving feature names for later use
feature_list = list(features.columns)
# Convert to numpy array
features = np.array(features)

In [51]:
# Split the data into training and testing sets
train_features, test_features, train_labels, test_labels = train_test_split(features, labels, test_size = 0.25, random_state = 42)

In [52]:
print('Training Features Shape:', train_features.shape)
print('Training Labels Shape:', train_labels.shape)
print('Testing Features Shape:', test_features.shape)
print('Testing Labels Shape:', test_labels.shape)

Training Features Shape: (525660, 89)
Training Labels Shape: (525660,)
Testing Features Shape: (175221, 89)
Testing Labels Shape: (175221,)


In [33]:
# # The baseline predictions are the historical averages
# baseline_preds = test_features[:, feature_list.index('hourly_counts')]
# # Baseline errors, and display average baseline error
# baseline_errors = abs(baseline_preds - test_labels)
# print('Average baseline error: ', round(np.mean(baseline_errors), 2))

In [53]:
# Instantiate model with 1000 decision trees
rf = RandomForestRegressor(n_estimators = 1000, random_state = 42)
# Train the model on training data
rf.fit(train_features, train_labels);

In [ ]:
# Use the forest's predict method on the test data
predictions = rf.predict(test_features)
# Calculate the absolute errors
errors = abs(predictions - test_labels)
# Print out the mean absolute error (mae)
print('Mean Absolute Error:', round(np.mean(errors), 2), 'degrees.')

In [ ]:
# Calculate mean absolute percentage error (MAPE)
mape = 100 * (errors / test_labels)
# Calculate and display accuracy
accuracy = 100 - np.mean(mape)
print('Accuracy:', round(accuracy, 2), '%.')

In [ ]:
# Pull out one tree from the forest
tree = rf.estimators_[5]

# Pull out one tree from the forest
tree = rf.estimators_[5]
# Export the image to a dot file
export_graphviz(tree, out_file = 'tree.dot', feature_names = feature_list, rounded = True, precision = 1)
# Use dot file to create a graph
(graph, ) = pydot.graph_from_dot_file('tree.dot')
# Write graph to a png file
graph.write_png('tree.png')

In [ ]:
# Limit depth of tree to 3 levels
rf_small = RandomForestRegressor(n_estimators=10, max_depth = 3)
rf_small.fit(train_features, train_labels)
# Extract the small tree
tree_small = rf_small.estimators_[5]
# Save the tree as a png image
export_graphviz(tree_small, out_file = 'small_tree.dot', feature_names = feature_list, rounded = True, precision = 1)
(graph, ) = pydot.graph_from_dot_file('small_tree.dot')

In [ ]:
# Get numerical feature importances
importances = list(rf.feature_importances_)
# List of tuples with variable and importance
feature_importances = [(feature, round(importance, 2)) for feature, importance in zip(feature_list, importances)]
# Sort the feature importances by most important first
feature_importances = sorted(feature_importances, key = lambda x: x[1], reverse = True)
# Print out the feature and importances 
[print('Variable: {:20} Importance: {}'.format(*pair)) for pair in feature_importances];